In [ ]:
from santa.eval import (
    generate_gift_recommendation,
    get_available_models,
    get_test_profiles,
    load_env_from_repo_root,
)

load_env_from_repo_root(".env", override=True)


In [ ]:
# Self-hosted vLLM smoke cell removed — use config/models.yaml + santa.eval.get_available_models.


In [ ]:
from santa.eval import (
    generate_gift_recommendation,
    get_available_models,
    get_test_profiles,
    load_env_from_repo_root,
)

load_env_from_repo_root(".env", override=True)


# Benchmark

In [ ]:
from santa.eval import run_benchmark_suite

models = get_available_models(
    include_openai=True, include_token_factory=True, include_self_hosted=True
)
profiles = get_test_profiles()
print(profiles)

In [ ]:
import pandas as pd
from santa.eval import results_to_dataframe

result_frames = []

for concurrency in [1, 5, 10, 20]:
    results = run_benchmark_suite(models, profiles, concurrency=concurrency)
    result_frames.append(results_to_dataframe(results, concurrency=concurrency))

df = pd.concat(result_frames, ignore_index=True)
df

## Plots

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (16, 12)

# Assuming df is your combined DataFrame with all concurrency levels
# If not, combine: df = pd.concat(result_frames, ignore_index=True)

# Create comprehensive analysis plots
fig = plt.figure(figsize=(18, 14))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# 1. Latency P99 vs Concurrency
ax1 = fig.add_subplot(gs[0, 0])
for model in df["model_name"].unique():
    model_data = df[df["model_name"] == model].sort_values("concurrency")
    ax1.plot(
        model_data["concurrency"],
        model_data["latency_p99_ms"],
        marker="o",
        label=model,
        linewidth=2.5,
        markersize=9,
        alpha=0.8,
    )

ax1.set_xlabel("Concurrency", fontsize=12, fontweight="bold")
ax1.set_ylabel("Latency P99 (ms)", fontsize=12, fontweight="bold")
ax1.set_title("Latency P99 vs Concurrency", fontsize=13, fontweight="bold")
ax1.legend(loc="best", frameon=True, shadow=True, fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_xscale("log", base=2)

# 2. Throughput vs Concurrency
ax2 = fig.add_subplot(gs[0, 1])
for model in df["model_name"].unique():
    model_data = df[df["model_name"] == model].sort_values("concurrency")
    ax2.plot(
        model_data["concurrency"],
        model_data["throughput_tokens_per_sec"],
        marker="s",
        label=model,
        linewidth=2.5,
        markersize=9,
        alpha=0.8,
    )

ax2.set_xlabel("Concurrency", fontsize=12, fontweight="bold")
ax2.set_ylabel("Throughput (tokens/sec)", fontsize=12, fontweight="bold")
ax2.set_title("Throughput vs Concurrency", fontsize=13, fontweight="bold")
ax2.legend(loc="best", frameon=True, shadow=True, fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xscale("log", base=2)

# 3. Cost per 1M Requests (Latency-Based) vs Concurrency
ax3 = fig.add_subplot(gs[1, 0])
for model in df["model_name"].unique():
    model_data = df[df["model_name"] == model].sort_values("concurrency")
    ax3.plot(
        model_data["concurrency"],
        model_data["cost_1m_requests"],
        marker="^",
        label=model,
        linewidth=2.5,
        markersize=9,
        alpha=0.8,
    )

ax3.set_xlabel("Concurrency", fontsize=12, fontweight="bold")
ax3.set_ylabel("Cost per 1M Requests ($)", fontsize=12, fontweight="bold")
ax3.set_title("Cost per 1M Requests (Latency-Based)", fontsize=13, fontweight="bold")
ax3.legend(loc="best", frameon=True, shadow=True, fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.set_xscale("log", base=2)

# 4. Cost per 1M Requests (Throughput-Based) vs Concurrency
ax4 = fig.add_subplot(gs[1, 1])
for model in df["model_name"].unique():
    model_data = df[df["model_name"] == model].sort_values("concurrency")
    # Filter out NaN and zero values for better visualization
    valid_data = model_data[model_data["cost_1m_requests_throughput_based"].notna()]
    if not valid_data.empty:
        # Show actual values, not rounded zeros
        ax4.plot(
            valid_data["concurrency"],
            valid_data["cost_1m_requests_throughput_based"],
            marker="D",
            label=model,
            linewidth=2.5,
            markersize=9,
            alpha=0.8,
        )
        # Add annotation for zero values explaining they're < $0.50
        zero_data = model_data[
            (model_data["cost_1m_requests_throughput_based"] == 0.0)
            | (model_data["cost_1m_requests_throughput_based"].isna())
        ]
        if not zero_data.empty:
            for _, row in zero_data.iterrows():
                ax4.annotate(
                    "< $0.50", xy=(row["concurrency"], 50), fontsize=8, alpha=0.6, ha="center"
                )

ax4.set_xlabel("Concurrency", fontsize=12, fontweight="bold")
ax4.set_ylabel("Cost per 1M Requests ($)", fontsize=12, fontweight="bold")
ax4.set_title("Cost per 1M Requests (Throughput-Based)", fontsize=13, fontweight="bold")
ax4.legend(loc="best", frameon=True, shadow=True, fontsize=9)
ax4.grid(True, alpha=0.3)
ax4.set_xscale("log", base=2)
ax4.set_yscale("log")  # Log scale to show the dramatic improvement

# 5. santa-deepseek-r1: Latency vs Throughput Cost Comparison
ax5 = fig.add_subplot(gs[2, :])
santa_data = df[df["model_name"] == "santa-deepseek-r1"].sort_values("concurrency")
if not santa_data.empty:
    ax5_twin = ax5.twinx()

    # Latency-based cost (left axis)
    line1 = ax5.plot(
        santa_data["concurrency"],
        santa_data["cost_1m_requests"],
        marker="o",
        label="Latency-Based Cost",
        linewidth=3,
        markersize=11,
        color="red",
        alpha=0.8,
    )

    # Throughput-based cost (left axis, but use different scale)
    valid_throughput = santa_data[santa_data["cost_1m_requests_throughput_based"].notna()]
    line2 = ax5.plot(
        valid_throughput["concurrency"],
        valid_throughput["cost_1m_requests_throughput_based"],
        marker="s",
        label="Throughput-Based Cost",
        linewidth=3,
        markersize=11,
        color="green",
        alpha=0.8,
    )

    # Throughput on right axis
    line3 = ax5_twin.plot(
        santa_data["concurrency"],
        santa_data["throughput_tokens_per_sec"],
        marker="^",
        label="Throughput (tokens/sec)",
        linewidth=2,
        markersize=9,
        color="blue",
        alpha=0.6,
        linestyle="--",
    )

    ax5.set_xlabel("Concurrency", fontsize=12, fontweight="bold")
    ax5.set_ylabel("Cost per 1M Requests ($)", fontsize=12, fontweight="bold", color="black")
    ax5_twin.set_ylabel("Throughput (tokens/sec)", fontsize=12, fontweight="bold", color="blue")
    ax5.set_title(
        "santa-deepseek-r1: Cost Comparison & Throughput Scaling", fontsize=14, fontweight="bold"
    )
    ax5.set_xscale("log", base=2)
    ax5.grid(True, alpha=0.3)

    # Combine legends
    lines = line1 + line2 + line3
    labels = [l.get_label() for l in lines]
    ax5.legend(lines, labels, loc="upper right", frameon=True, shadow=True)

plt.suptitle(
    "LLM Benchmark Analysis: Performance vs Cost Across Concurrency Levels",
    fontsize=16,
    fontweight="bold",
    y=0.995,
)
plt.tight_layout()
plt.show()

# Print detailed analysis
print("\n" + "=" * 80)
print("DETAILED ANALYSIS")
print("=" * 80)

# Cost efficiency analysis
print("\n📊 Cost Efficiency Analysis (Throughput-Based):")
print("-" * 80)
for model in df["model_name"].unique():
    model_data = df[df["model_name"] == model].sort_values("concurrency")
    best_throughput = model_data.loc[model_data["throughput_tokens_per_sec"].idxmax()]
    best_cost = model_data[
        model_data["cost_1m_requests_throughput_based"].notna()
        & (model_data["cost_1m_requests_throughput_based"] > 0)
    ]

    if not best_cost.empty:
        best_cost_row = best_cost.loc[best_cost["cost_1m_requests_throughput_based"].idxmin()]
        print(f"\n{model}:")
        print(
            f"  Best throughput: {best_throughput['throughput_tokens_per_sec']:.1f} tok/s at concurrency={best_throughput['concurrency']}"
        )
        if not best_cost.empty:
            print(
                f"  Best cost: ${best_cost_row['cost_1m_requests_throughput_based']:.0f} per 1M requests at concurrency={best_cost_row['concurrency']}"
            )
            print(f"  Latency P99 at best cost: {best_cost_row['latency_p99_ms']:.1f} ms")